In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import format_number, col, mean, max, min, year, month, avg
from pyspark.sql.types import DoubleType

# 初始化 Spark 会话
spark_session = SparkSession.builder \
    .appName("DetailedWalmartStockAnalysis") \
    .getOrCreate()

# 数据加载：读取 CSV 文件，并自动推断列数据类型
data_path = "data\walmart_stock.csv"
stock_df = spark_session.read.csv(data_path, header=True, inferSchema=True)

# 输出数据集的列名
print("列名:", stock_df.columns)

# 显示数据集的结构
stock_df.printSchema()

# 显示数据集的前五条记录
stock_df.show(5)

# 描述性统计：提供基本统计概览
stats_summary_df = stock_df.summary()

# 对统计结果进行格式化，以便更清晰地呈现
formatted_stats_df = stats_summary_df.select(
    col("summary"),
    format_number(col("Open").cast(DoubleType()), 2).alias("Formatted Open"),
    format_number(col("High").cast(DoubleType()), 2).alias("Formatted High"),
    format_number(col("Low").cast(DoubleType()), 2).alias("Formatted Low"),
    format_number(col("Close").cast(DoubleType()), 2).alias("Formatted Close"),
    format_number(col("Volume").cast(DoubleType()), 2).alias("Formatted Volume"),
    format_number(col("Adj Close").cast(DoubleType()), 2).alias("Formatted Adj Close")
)

formatted_stats_df.show()

# 添加新列：高/成交量比例（HV Ratio），并展示
hv_df = stock_df.withColumn("HV Ratio", col("High") / col("Volume"))
hv_df.select("Date", "High", "Volume", "HV Ratio").show(5)

# 查询最高价最高的一天
highest_day = stock_df.orderBy(stock_df["High"].desc()).first()
print(f"最高价最高的那一天: {highest_day['Date']}, 最高价: {highest_day['High']}")

# 计算平均收盘价
avg_close_price = stock_df.agg(mean("Close")).first()[0]
print(f"平均收盘价: {avg_close_price:.2f}")

# 成交量的极值：最大值和最小值
max_volume = stock_df.agg(max("Volume")).first()[0]
min_volume = stock_df.agg(min("Volume")).first()[0]
print(f"成交量最大值: {max_volume}, 最小值: {min_volume}")

# 计算收盘价低于 60 的总天数
count_below_60 = stock_df.filter(stock_df["Close"] < 60).count()
print(f"收盘价低于 60 的总天数: {count_below_60}")

# 计算最高价超过 80 的天数百分比
total_days_count = stock_df.count()
days_above_80_count = stock_df.filter(stock_df["High"] > 80).count()
percentage_above_80 = (days_above_80_count / total_days_count) * 100
print(f"最高价超过 80 的天数占比: {percentage_above_80:.2f}%")

# 最高价与成交量的皮尔逊相关系数
high_volume_corr = stock_df.stat.corr("High", "Volume")
print(f"最高价与成交量的相关系数: {high_volume_corr:.2f}")

# 按年度统计最高价
annual_high_df = stock_df.withColumn("Year", year("Date")).groupBy("Year").agg(max("High").alias("Annual Max High")).orderBy("Year")
annual_high_df.show()

# 按月统计平均收盘价
monthly_avg_df = stock_df.withColumn("Month", month("Date")).groupBy("Month").agg(avg("Close").alias("Monthly Avg Close")).orderBy("Month")
monthly_avg_df.show()

# 结束 Spark 会话
spark_session.stop()


列名: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Adj Close']
root
 |-- Date: timestamp (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: integer (nullable = true)
 |-- Adj Close: double (nullable = true)

+-------------------+------------------+---------+---------+------------------+--------+------------------+
|               Date|              Open|     High|      Low|             Close|  Volume|         Adj Close|
+-------------------+------------------+---------+---------+------------------+--------+------------------+
|2012-01-03 00:00:00|         59.970001|61.060001|59.869999|         60.330002|12668800|52.619234999999996|
|2012-01-04 00:00:00|60.209998999999996|60.349998|59.470001|59.709998999999996| 9593300|         52.078475|
|2012-01-05 00:00:00|         59.349998|59.619999|58.369999|         59.419998|12768200|         51.825539|
|2012-01-06 00: